# Interprétation du modèle v3 — corrélations et SHAP

Le modèle atteint une PR-AUC de 0,0177, soit **73,4× le hasard**. Reste la
question qui compte autant que le score : **sur quoi s'appuie-t-il ?**

Deux outils, dans cet ordre — et l'ordre n'est pas négociable.

---

### Pourquoi la matrice de corrélation AVANT SHAP

Le v3 a montré quelque chose de troublant : `taux_commune_lisse` a pris 25
points d'importance pendant que `feux_commune_365j` en perdait 24. Les deux
disent la même chose — « cette commune brûle-t-elle ».

C'est le piège classique de l'interprétation d'un modèle à arbres. **Quand deux
features portent la même information, l'arbre en choisit UNE à chaque nœud et
l'autre paraît inutile.** SHAP hérite du problème : il attribue la contribution
à celle qui a été retenue, pas aux deux.

Lire un graphique SHAP sans la matrice de corrélation sous les yeux, c'est
risquer de conclure qu'une variable ne sert à rien alors qu'elle est simplement
doublée.

### Ce que SHAP dit de plus que l'importance

| | Répond à |
|---|---|
| importance XGBoost | à quel point cette feature sert à **découper** |
| **SHAP** | de combien cette feature a **déplacé le score de cette ligne**, et dans quel sens |

SHAP est *signé* et *local*. C'est ce qui permet de dire « pour cette commune,
ce jour-là, le FWI a ajouté 0,4 et le maquis 0,9 » — et donc de justifier une
alerte devant un opérationnel.

⚠️ **TreeSHAP est EXACT** sur un modèle à arbres, ce n'est pas une
approximation par échantillonnage comme KernelSHAP. Le seul arbitrage porte sur
le nombre de lignes expliquées, jamais sur la qualité de l'explication.

### Sur quelles lignes

38 M lignes sont hors de question. Deux échantillons, deux questions :

| Échantillon | Ce qu'il montre |
|---|---|
| **aléatoire** | ce que fait le modèle une journée ordinaire — représentatif, mais quasi aucun feu (0,024 %) |
| **sommet** | les 60 000 lignes les mieux notées, celles qu'un opérationnel regarderait vraiment |

⚠️ Les deux n'ont pas le même taux de positifs : **les amplitudes SHAP ne se
comparent pas d'un échantillon à l'autre.** Les classements et les formes, si.

## 1. Chargement

In [ ]:
# ── enregistrement automatique des figures ──
# chaque plt.show() écrit aussi un PNG dans figures/interpretation/
import sys
from pathlib import Path

for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "src" / "tvfed").is_dir():
        sys.path.insert(0, str(_p / "src"))
        break

from tvfed.figures import activer

activer("interpretation")

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

for p in (Path.cwd(), *Path.cwd().parents):
    if (p / "src" / "tvfed").is_dir():
        sys.path.insert(0, str(p / "src"))
        RACINE = p
        break

INK, MUTED, GRID = "#0b0b0b", "#898781", "#e1e0d9"
BLEU, ORANGE, ROUGE, VERT, VIOLET = "#2a78d6", "#eb6834", "#e34948", "#1baf7a", "#4a3aa7"
GRIS = "#c3c2b7"
plt.rcParams.update({"figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
                     "font.size": 9, "axes.edgecolor": "#c3c2b7", "text.color": INK})

PROC = RACINE / "data" / "processed"
CORR = pd.read_parquet(PROC / "correlations_v3.parquet")
RED = pd.read_csv(PROC / "paires_redondantes.csv")
COLS = pd.read_csv(PROC / "shap_colonnes.csv").feature.to_list()

SH = {n: np.load(PROC / f"shap_{n}.npy") for n in ("alea", "sommet")}
XS = {n: pd.read_parquet(PROC / f"shap_{n}_X.parquet") for n in ("alea", "sommet")}

# les six blocs thématiques, repris des notebooks précédents
BLOCS = {
    "clustering": (["cluster_id", "taux_cluster_lisse", "taux_commune_lisse",
                    "ratio_commune_cluster"], VERT),
    "historique commune": (["feux_commune_7j", "feux_commune_30j", "feux_commune_90j",
                            "feux_commune_365j", "jours_depuis_dernier_feu"], VIOLET),
    "météo": (["fwi", "ffmc", "dmc", "dc", "bui", "isi", "kbdi", "erc",
               "danger_effis", "fwi_j1", "ffmc_j1"], ORANGE),
    "géographie": (["lat", "lon", "distance_cote_km", "altitude_moy",
                    "amplitude_altitude", "log_superficie"], BLEU),
    "végétation": ([c for c in COLS if c.startswith("part_")] + ["clc_millesime"], VERT),
    "présence humaine": (["log_population", "log_densite", "grille_densite"], ROUGE),
}
BLOCS["végétation"] = ([c for c in COLS if c.startswith("part_")] + ["clc_millesime"],
                       "#7cc4a0")


def bloc_de(f):
    for nom, (feats, _) in BLOCS.items():
        if f in feats:
            return nom
    return "calendrier"


COUL_BLOC = {n: c for n, (_, c) in BLOCS.items()} | {"calendrier": GRIS}

print(f"{len(COLS)} features")
print(f"corrélations : {len(RED)} paires |ρ| ≥ 0,90")
for n in ("alea", "sommet"):
    d = XS[n]
    print(f"SHAP {n:7s} {SH[n].shape}  {d.y.sum():>5,} feux "
          f"({d.y.mean():.3%})  score médian {d._p.median():.5f}")


## 2. Qui dit la même chose que qui ?

Corrélation de **Spearman** et non de Pearson : plusieurs features sont très
asymétriques — les comptages de feux, `jours_depuis_dernier_feu`, les parts
CORINE écrasées sur zéro. Une corrélation de rang mesure la co-monotonie sans
supposer de linéarité, ce qui correspond à ce qu'un arbre exploite réellement.

In [ ]:
"""FIG 1 — Quelles features disent la même chose ?

À lire AVANT tout graphique SHAP.

Quand deux features portent la même information, un arbre en choisit UNE à
chaque nœud et l'autre paraît inutile. SHAP hérite du problème : il attribue
la contribution à celle qui a été retenue. Sans cette matrice sous les yeux,
on conclurait qu'une variable ne sert à rien alors qu'elle est simplement
doublée.

Corrélation de SPEARMAN et non de Pearson : plusieurs features sont très
asymétriques — les comptages de feux, `jours_depuis_dernier_feu`, les parts
CORINE écrasées sur zéro. Une corrélation de rang mesure la co-monotonie sans
supposer de linéarité, ce qui correspond à ce qu'un arbre exploite réellement.
"""
# les features regroupées par bloc, pour que la matrice soit lisible
ordre = [f for b in ("clustering", "historique commune", "météo", "végétation",
                     "géographie", "présence humaine", "calendrier")
         for f in COLS if bloc_de(f) == b]
M = CORR.loc[ordre, ordre]

fig = plt.figure(figsize=(15.5, 6.6))
gs = fig.add_gridspec(1, 2, width_ratios=[1.05, 1.15], wspace=.22)

# ── (a) la matrice ───────────────────────────────────────────────────────
ax0 = fig.add_subplot(gs[0])
im = ax0.imshow(M.to_numpy(), cmap="RdBu_r", vmin=-1, vmax=1)
ax0.set_xticks(range(len(ordre)))
ax0.set_xticklabels(ordre, rotation=90, fontsize=5.2)
ax0.set_yticks(range(len(ordre)))
ax0.set_yticklabels(ordre, fontsize=5.2)
ax0.tick_params(colors=MUTED, length=0)
ax0.spines[:].set_visible(False)

# séparateurs entre blocs + étiquette de bloc
bornes, courant = [], None
for i, f in enumerate(ordre):
    b = bloc_de(f)
    if b != courant:
        bornes.append((i, b)); courant = b
for i, (d, b) in enumerate(bornes):
    fin = bornes[i + 1][0] if i + 1 < len(bornes) else len(ordre)
    for a in (ax0.axhline, ax0.axvline):
        a(d - .5, color=INK, lw=1.1)
    ax0.text(-7.5, (d + fin - 1) / 2, b.replace(" ", "\n"), fontsize=6.5,
             ha="right", va="center", color=COUL_BLOC[b], weight="bold")
cb = fig.colorbar(im, ax=ax0, fraction=.040, pad=.02)
cb.set_label("corrélation de Spearman", fontsize=8.5)
cb.ax.tick_params(labelsize=8, colors=MUTED); cb.outline.set_visible(False)
ax0.set_title("Les 52 features, regroupées par bloc",
              fontsize=11.5, weight="bold", loc="left")

# ── (b) les paires redondantes ───────────────────────────────────────────
ax1 = fig.add_subplot(gs[1])
R = RED.head(14).iloc[::-1]
etq = [f"{a}\n↔ {b}" for a, b in zip(R.feature_a, R.feature_b)]
c = [ROUGE if abs(v) >= .99 else ORANGE if abs(v) >= .95 else GRIS
     for v in R.spearman]
ax1.barh(range(len(R)), R.spearman.abs(), color=c, edgecolor="#fcfcfb",
         linewidth=.8, height=.74)
for i, v in enumerate(R.spearman):
    ax1.text(abs(v) - .012, i, f"{v:+.3f}", va="center", ha="right",
             fontsize=8.5, color="#fcfcfb", weight="bold")
ax1.set_yticks(range(len(R))); ax1.set_yticklabels(etq, fontsize=7)
ax1.set_xlim(.85, 1.005)
ax1.set_xlabel("|corrélation de Spearman|")
ax1.axvline(.90, color=INK, ls="--", lw=1.2)
ax1.text(.902, -.85, "seuil de redondance", fontsize=8, color=MUTED)
ax1.set_title(f"{len(RED)} paires au-dessus de 0,90",
              fontsize=11.5, weight="bold", loc="left")
ax1.grid(axis="x", color=GRID, lw=.7); ax1.set_axisbelow(True)
ax1.spines[["top", "right"]].set_visible(False); ax1.tick_params(colors=MUTED)

fig.suptitle("Redondance entre features — à lire avant tout graphique SHAP",
             fontsize=13, weight="bold", x=.005, ha="left", y=1.0)
plt.show()

print(f"{len(RED)} paires au-dessus de |ρ| = 0,90 :\n")
for r in RED.itertuples():
    print(f"  {r.spearman:+.3f}   {r.feature_a:24s} ↔ {r.feature_b}")
print()
print("→ Trois familles de redondance, et elles n'ont PAS le même statut :")
print()
print("  1. REDONDANCE PAR DÉFINITION — rien à corriger, c'est la physique.")
print("     Les indices CEMS se calculent les uns à partir des autres :")
print("     BUI se déduit de DMC et DC, ISI de FFMC et du vent, FWI de BUI")
print("     et ISI. De même `part_combustible` CONTIENT `part_foret` (tous")
print("     les postes CLC_3* contre les seuls 311/312/313). Leur corrélation")
print("     n'est pas une propriété de nos données, c'est leur définition.")
print()
print("  2. REDONDANCE D'ENCODAGE — la même date écrite deux fois.")
print("     `doy` ↔ `mois`, `sin_doy` ↔ `sin_mois`, `cos_doy` ↔ `cos_mois`.")
print("     Coût nul en performance, mais l'interprétation est diluée sur")
print("     deux colonnes qui disent strictement la même chose.")
print()
print("  3. REDONDANCE DE FOND — la seule qui pose une vraie question.")
print("     `feux_commune_365j` ↔ `jours_depuis_dernier_feu` à ρ = −0,999 :")
print("     à l'ordre près, LA MÊME VARIABLE. 80 % des lignes n'ont aucun feu")
print("     sur 365 jours, donc les deux sont collées à leur valeur extrême")
print("     sur la même masse de lignes.")
print("     `taux_cluster_lisse` ↔ `taux_commune_lisse` à 0,930 : attendu,")
print("     le second est construit en rappelant vers le premier.")
print()
print("⚠️ CE QUE ÇA IMPLIQUE POUR LIRE LA SUITE")
print("   Sur ces paires, importance et SHAP sont RÉPARTIS ARBITRAIREMENT")
print("   entre les deux membres — l'arbre en prend une au hasard des seuils")
print("   qu'il teste. Voir une feature à 1 % ne prouve pas qu'elle est")
print("   inutile : sa jumelle a peut-être capté toute la contribution.")
print("   Il faut donc lire les BLOCS autant que les features individuelles.")


## 3. Ce qui déplace le score

L'unité est le **log-odds** : c'est l'échelle sur laquelle le modèle
additionne ses contributions avant de les convertir en probabilité. Une
contribution de +1 double à peu près la cote de l'événement.

Les couleurs reprennent les six blocs thématiques des notebooks précédents.

In [ ]:
"""FIG 2 — Ce qui pousse le score vers le haut, et de combien.

L'importance de XGBoost dit « à quel point cette feature sert à découper ».
SHAP dit autre chose, et de plus utile : **de combien cette feature a déplacé
le score de CETTE ligne-là, dans quel sens**.

Deux populations, deux questions distinctes :

  ALÉATOIRE   ce que fait le modèle sur une journée ordinaire (0,024 % de feux)
  SOMMET      les 60 000 lignes les mieux notées — celles qu'un opérationnel
              regarderait vraiment

⚠️ Les deux échantillons n'ont pas le même taux de positifs : les AMPLITUDES
ne se comparent pas d'un panneau à l'autre. Les classements et les formes, si.
"""
N_TOP = 14

fig, ax = plt.subplots(1, 2, figsize=(15, 6.2))

for k, (nom, titre) in enumerate([
        ("alea", "Sur une journée ordinaire"),
        ("sommet", "Sur les 60 000 lignes les mieux notées")]):
    v, X = SH[nom], XS[nom]
    moy = pd.Series(np.abs(v).mean(0), index=COLS).sort_values(ascending=False)
    top = moy.head(N_TOP).iloc[::-1]
    c = [COUL_BLOC[bloc_de(f)] for f in top.index]

    ax[k].barh(range(len(top)), top.to_numpy(), color=c,
               edgecolor="#fcfcfb", linewidth=.8, height=.74)
    for i, val in enumerate(top):
        ax[k].text(val + top.max() * .02, i, f"{val:.3f}", va="center",
                   fontsize=8.5, color=MUTED)
    ax[k].set_yticks(range(len(top)))
    ax[k].set_yticklabels([f.replace("_", " ") for f in top.index], fontsize=8.5)
    ax[k].set_xlim(0, top.max() * 1.16)
    ax[k].set_xlabel("|SHAP| moyen — déplacement du score, en log-odds")
    ax[k].set_title(f"{titre}\n{X.y.sum():,} feux sur {len(X):,} lignes "
                    f"({X.y.mean():.2%})",
                    fontsize=11.5, weight="bold", loc="left")
    ax[k].grid(axis="x", color=GRID, lw=.7); ax[k].set_axisbelow(True)
    ax[k].spines[["top", "right"]].set_visible(False)
    ax[k].tick_params(colors=MUTED)

# légende des blocs, une seule fois
from matplotlib.patches import Patch
vus = {bloc_de(f) for n in SH for f in
       pd.Series(np.abs(SH[n]).mean(0), index=COLS).nlargest(N_TOP).index}
ax[1].legend(handles=[Patch(facecolor=COUL_BLOC[b], label=b) for b in
                      sorted(vus, key=lambda b: -sum(
                          bloc_de(f) == b for f in COLS))],
             frameon=False, fontsize=8.5, loc="lower right", title="bloc",
             title_fontsize=8.5)

fig.suptitle("SHAP — ce qui déplace vraiment le score, et sur quelle population",
             fontsize=13, weight="bold", x=.005, ha="left", y=1.0)
plt.tight_layout(); plt.show()

for nom in ("alea", "sommet"):
    moy = pd.Series(np.abs(SH[nom]).mean(0), index=COLS).sort_values(ascending=False)
    par_bloc = moy.groupby(pd.Series(COLS, index=COLS).map(bloc_de)).sum()
    par_bloc = (100 * par_bloc / par_bloc.sum()).sort_values(ascending=False)
    print(f"\n{nom.upper()} — répartition par bloc (% du |SHAP| total)")
    for b, p in par_bloc.items():
        print(f"   {b:22s} {p:5.1f} %")

a = pd.Series(np.abs(SH["alea"]).mean(0), index=COLS)
s = pd.Series(np.abs(SH["sommet"]).mean(0), index=COLS)
ra, rs = a.rank(ascending=False), s.rank(ascending=False)
# ⚠️ rang 1 = plus importante : GAGNER en importance, c'est voir son numéro
# de rang DIMINUER. Le gain vaut donc ra − rs, positif quand la feature monte.
gain = (ra - rs).sort_values(ascending=False)
print(f"\n{'─' * 62}")
print("CE QUI CHANGE ENTRE LA JOURNÉE ORDINAIRE ET LE SOMMET")
print(f"{'─' * 62}")
print("features qui MONTENT quand on regarde le sommet :")
for f in gain.head(6).index:
    print(f"   {f:24s} rang {int(ra[f]):>2} → {int(rs[f]):>2}   ({gain[f]:+.0f})")
print("\nfeatures qui DESCENDENT :")
for f in gain.tail(6).index[::-1]:
    print(f"   {f:24s} rang {int(ra[f]):>2} → {int(rs[f]):>2}   ({gain[f]:+.0f})")
print()
print("→ Sur une journée ordinaire, le modèle tourne à la MÉTÉO et à la")
print("  GÉOGRAPHIE : il n'a rien d'autre à dire, tout est à zéro.")
print("  Sur les lignes qu'il note le plus haut, ce sont le CLUSTERING et")
print("  l'HISTORIQUE qui prennent la main. Autrement dit : la météo trie les")
print("  jours, le territoire trie les communes — et c'est bien au sommet,")
print("  là où la décision se prend, que le territoire pèse le plus.")


## 4. La thèse du projet, mise à l'épreuve

Tout le cadrage tient en une phrase : **la météo dit QUAND, le territoire dit
OÙ, et c'est leur croisement qui prédit.**

C'est une affirmation testable, et SHAP est exactement l'outil pour la tester.
Si elle est juste, le modèle ne doit pas traiter le FWI de la même façon
partout : une journée à FWI 30 ne devrait pas peser pareil sur un maquis varois
et sur une plaine céréalière.

Si les trois courbes du panneau central se superposent, la thèse est fausse et
le modèle traite la météo indépendamment du lieu. Si elles divergent, il a bien
appris le croisement — et toute la construction de la base se justifie.

In [ ]:
"""FIG 3 — La thèse du projet, mise à l'épreuve (et un piège évité de justesse).

Le cadrage tient en une phrase : **la météo dit QUAND, le territoire dit OÙ,
et c'est leur croisement qui prédit.** C'est testable : si elle est juste, une
journée à FWI 30 ne doit pas peser pareil sur un maquis varois et sur une
plaine céréalière.

⚠️ MAIS LE TEST NE PEUT PAS SE FAIRE SUR N'IMPORTE QUELLE POPULATION.

Fait sur l'échantillon « sommet », il donne le signe INVERSE — et c'est un
artefact. Le sommet est sélectionné sur le score du modèle : parmi les lignes
les mieux notées, une commune peu exposée ne peut s'y trouver que si la météo
fait tout le travail. Conditionner sur le score, c'est conditionner sur un
descendant commun des deux causes qu'on veut comparer — un *collider*. Le biais
est mécanique et il renverse la conclusion.

Le test valide se fait donc sur l'échantillon ALÉATOIRE. Le panneau de droite
montre le renversement, parce qu'il vaut mieux le regarder en face que de
l'ignorer.
"""
BORNES = [(0, 5.2, "très\nfaible"), (5.2, 11.2, "faible"), (11.2, 21.3, "modéré"),
          (21.3, 38, "élevé"), (38, 1e9, "très\nélevé")]


def tableau(nom):
    """Contribution médiane du FWI, par tranche de FWI et de risque territorial."""
    v, X = SH[nom], XS[nom]
    s = pd.DataFrame(v, columns=COLS).fwi.to_numpy()
    terr, fwi = X.taux_commune_lisse.to_numpy(), X.fwi.to_numpy()
    b1, b2 = np.quantile(terr, [1 / 3, 2 / 3])
    grp = [terr <= b1, (terr > b1) & (terr <= b2), terr > b2]
    out = []
    for lo, hi, lab in BORNES:
        mf = (fwi >= lo) & (fwi < hi)
        out.append([np.median(s[mf & g]) if (mf & g).sum() >= 20 else np.nan
                    for g in grp])
    return pd.DataFrame(out, index=[b[2] for b in BORNES],
                        columns=["peu exposé", "moyen", "très exposé"])


TA, TS = tableau("alea"), tableau("sommet")
COUL = ["#9ec9f0", BLEU, "#12406e"]

fig, ax = plt.subplots(1, 3, figsize=(16, 4.9))

# ── (a) le territoire seul : quelle forme ? ─────────────────────────────
v, X = SH["alea"], XS["alea"]
S = pd.DataFrame(v, columns=COLS)
x = X.taux_commune_lisse.to_numpy() * 100
ax[0].scatter(x, S.taux_commune_lisse, s=5, alpha=.18, color=VERT,
              edgecolors="none")
d = pd.DataFrame({"x": x, "s": S.taux_commune_lisse})
m = d.groupby(pd.qcut(d.x, 24, duplicates="drop"), observed=True).median()
ax[0].plot(m.x, m.s, lw=2.6, color="#0d7a52")
ax[0].axhline(0, color=INK, lw=1.1)
ax[0].set_xlabel("risque lissé de la commune (%)")
ax[0].set_ylabel("contribution au score (log-odds)")
ax[0].set_title("Le territoire, seul", fontsize=11.5, weight="bold", loc="left")

# ── (b) et (c) — même échelle, sinon la comparaison est trompeuse ───────
x = np.arange(len(BORNES))
lo = min(TA.to_numpy().min(), TS.to_numpy().min())
hi = max(TA.to_numpy().max(), TS.to_numpy().max())
marge = (hi - lo) * .30

for k, (T, titre, note, coul_note) in enumerate([
    (TA, "VALIDE — échantillon aléatoire",
     None, INK),
    (TS, "TROMPEUR — même calcul, échantillon « sommet »",
     None, ROUGE),
]):
    a = ax[k + 1]
    for i, c in enumerate(T.columns):
        a.plot(x, T[c].to_numpy(), lw=2.6, marker="o", ms=6, color=COUL[i],
               label=c)
    a.axhline(0, color=INK, lw=1.1)
    a.set_xticks(x)
    a.set_xticklabels([b[2].replace("\n", " ") for b in BORNES], fontsize=8)
    a.set_xlabel("FWI du jour")
    a.set_ylim(lo - marge * .35, hi + marge)
    a.set_title(titre, fontsize=11.5, weight="bold", loc="left")
ax[1].set_ylabel("contribution du FWI (log-odds)")
ax[1].legend(frameon=False, fontsize=8.5, loc="lower right", title="territoire",
             title_fontsize=8.5)

e = (TA["très exposé"] - TA["peu exposé"]).iloc[1:]
es = (TS["très exposé"] - TS["peu exposé"]).iloc[1:]
ax[1].text(.03, .97, f"à FWI comparable, le territoire exposé\n"
                     f"AJOUTE {e.min():+.3f} à {e.max():+.3f} au poids de la météo\n"
                     f"→ la météo compte PLUS là où ça peut brûler",
           transform=ax[1].transAxes, va="top", fontsize=8.5, color=INK,
           bbox=dict(boxstyle="round,pad=.5", fc="#fcfcfb", ec=GRID))
ax[2].text(.03, .97, f"l'ordre s'INVERSE ({es.min():+.3f} à {es.max():+.3f}).\n"
                     f"Artefact de sélection : filtrer sur le score,\n"
                     f"c'est conditionner sur un descendant commun\n"
                     f"des deux causes comparées — un collider.",
           transform=ax[2].transAxes, va="top", fontsize=8.5, color=ROUGE,
           bbox=dict(boxstyle="round,pad=.5", fc="#fcfcfb", ec=ROUGE))

for a in ax:
    a.grid(color=GRID, lw=.7); a.set_axisbelow(True)
    a.spines[["top", "right"]].set_visible(False); a.tick_params(colors=MUTED)
fig.suptitle("Le modèle croise-t-il météo et territoire ? — et sur quelle "
             "population a-t-on le droit de le demander",
             fontsize=13, weight="bold", x=.005, ha="left", y=1.02)
plt.tight_layout(); plt.show()

print("Contribution MÉDIANE du FWI au score (log-odds)\n")
for nom, T in (("ÉCHANTILLON ALÉATOIRE — le test valide", TA),
               ("ÉCHANTILLON « SOMMET » — sélectionné sur le score", TS)):
    print(f"{nom}")
    t = T.copy()
    t["écart"] = t["très exposé"] - t["peu exposé"]
    print(t.to_string(float_format=lambda v: f"{v:+.3f}"))
    print()

print("─" * 66)
print("CE QUE ÇA DIT")
print("─" * 66)
print("Sur l'échantillon ALÉATOIRE — le seul valide — l'écart est POSITIF dès")
print("que le FWI dépasse le seuil « très faible » : à météo identique, le")
print(f"modèle accorde {e.min():+.3f} à {e.max():+.3f} de plus au FWI quand le territoire")
print("est exposé. La météo compte davantage là où il y a de quoi brûler.")
print("La thèse du projet est vérifiée, et elle est MESURÉE, pas postulée.")
print()
print("Une nuance qui a du sens : dans la tranche « très faible », l'écart")
print(f"s'inverse ({TA['très exposé'].iloc[0] - TA['peu exposé'].iloc[0]:+.3f}). Le modèle dit que même dans le Var,")
print("un jour à FWI quasi nul ne brûle pas — et il le dit plus fort là-bas")
print("qu'ailleurs. C'est cohérent : le combustible ne suffit pas.")
print()
print("⚠️ SUR L'ÉCHANTILLON « SOMMET », L'ORDRE S'INVERSE COMPLÈTEMENT.")
print("   Ce n'est pas un résultat, c'est un biais de sélection. Parmi les")
print("   lignes les mieux notées, une commune peu exposée ne peut se trouver")
print("   là QUE si la météo est extrême — donc sa contribution météo y paraît")
print("   plus forte. Conditionner sur le score du modèle revient à")
print("   conditionner sur un descendant commun des deux causes qu'on compare.")
print()
print("   → Règle : une analyse d'interaction se fait sur une population")
print("     REPRÉSENTATIVE, jamais sur une population filtrée par le modèle")
print("     lui-même. Le panneau de droite est là pour s'en souvenir.")


## 5. Synthèse

### Le modèle n'est pas le même selon l'heure de la journée

C'est le résultat le plus net, et il n'apparaît qu'en séparant les deux
populations. Répartition du |SHAP| total par bloc :

| Bloc | Journée ordinaire | Sommet du classement |
|---|---|---|
| **clustering** | 21,4 % | **34,9 %** |
| météo | **30,3 %** | 22,0 % |
| historique commune | 4,0 % | **20,1 %** |
| géographie | **27,0 %** | 11,1 % |
| végétation | 6,8 % | 5,4 % |
| calendrier | 7,7 % | 4,2 % |
| présence humaine | 2,8 % | 2,4 % |

Sur une journée ordinaire, le modèle tourne à la **météo** et à la
**géographie** — `lat` y est même la feature n°1 (|SHAP| 1,03), ce qui est une
façon coûteuse de dire « le Sud brûle plus ». Il n'a rien d'autre à dire : tout
l'historique est à zéro sur 99,98 % des lignes.

Sur les lignes qu'il note le plus haut — celles qu'un opérationnel regarderait
— ce sont le **clustering** et l'**historique** qui prennent la main.
`taux_commune_lisse` y pèse 1,58, soit trois fois le suivant. `lat` tombe au
6ᵉ rang.

**La météo trie les jours, le territoire trie les communes.** Et c'est bien au
sommet, là où la décision se prend, que le territoire pèse le plus.

Les mouvements de rang le disent aussi :

| Feature | Rang ordinaire → sommet |
|---|---|
| `feux_commune_30j` | 41 → **10** |
| `feux_commune_90j` | 38 → **12** |
| `jours_depuis_dernier_feu` | 15 → **2** |
| `part_combustible` | 21 → 35 |
| `altitude_moy` | 25 → 38 |
| `mois` | 27 → 39 |

### La thèse du projet : vérifiée, et mesurée

Contribution médiane du FWI au score, par tranche de FWI et de risque
territorial — **sur l'échantillon aléatoire** :

| FWI | peu exposé | moyen | très exposé | écart |
|---|---|---|---|---|
| très faible | −0,358 | −0,386 | −0,429 | −0,071 |
| faible | +0,046 | +0,062 | +0,068 | **+0,022** |
| modéré | +0,105 | +0,120 | +0,132 | **+0,027** |
| élevé | +0,196 | +0,210 | +0,242 | **+0,046** |
| très élevé | +0,375 | +0,343 | +0,385 | +0,010 |

À météo identique, le modèle accorde **plus de poids au FWI quand le territoire
est exposé**. La météo compte davantage là où il y a de quoi brûler. C'est
exactement le croisement pour lequel la base a été construite — et il est
*mesuré*, pas postulé.

La nuance de la première ligne a du sens : à FWI quasi nul, l'écart s'inverse
(−0,071). Le modèle dit que même dans le Var, un jour sans danger ne brûle pas
— et il le dit plus fort là-bas qu'ailleurs. Le combustible ne suffit pas.

### ⚠️ Le piège dans lequel je suis tombé

Le même calcul sur l'échantillon « sommet » donne des écarts **négatifs**
(−0,014 à −0,076) : l'ordre s'inverse complètement, et la conclusion aurait été
l'exacte opposée.

Ce n'est pas un résultat, c'est un **biais de sélection**. Le sommet est filtré
sur le score du modèle. Parmi les lignes les mieux notées, une commune peu
exposée ne peut s'y trouver *que* si la météo est extrême — sa contribution
météo y paraît donc mécaniquement plus forte. Conditionner sur le score, c'est
conditionner sur un descendant commun des deux causes qu'on compare : un
**collider**.

→ **Règle** : une analyse d'interaction se fait sur une population
*représentative*, jamais sur une population filtrée par le modèle lui-même. Le
panneau de droite de la figure 3 est conservé pour s'en souvenir.

### Ce que la redondance interdit de conclure

13 paires de features au-dessus de |ρ| = 0,90, en trois familles de statut très
différent :

| Famille | Exemples | Faut-il agir ? |
|---|---|---|
| **par définition** | `dmc` ↔ `bui` (0,991), `ffmc` ↔ `isi` (0,957), `part_foret` ⊂ `part_combustible` (0,911) | non — c'est la physique des indices CEMS et la nomenclature CORINE |
| **d'encodage** | `doy` ↔ `mois` (0,996), `sin_doy` ↔ `sin_mois` (0,954) | la même date écrite deux fois ; coût nul, interprétation diluée |
| **de fond** | `feux_commune_365j` ↔ `jours_depuis_dernier_feu` (**−0,999**) | à l'ordre près, **la même variable** |

Sur ces paires, l'importance et le SHAP sont **répartis arbitrairement** entre
les deux membres : l'arbre en retient une au hasard des seuils qu'il teste.
Voir une feature à 1 % ne prouve donc pas qu'elle est inutile.

→ **C'est pourquoi la lecture par BLOC du tableau ci-dessus est plus solide que
la lecture feature par feature.**

### Ce qui reste à faire

1. **MLP**, la troisième famille exigée par l'énoncé
2. **Calibration du v3** sur la validation, puis **évaluation test — une seule
   fois**
3. **Application Streamlit** : les valeurs SHAP par ligne permettent de
   justifier chaque alerte (« cette commune, ce jour : territoire +1,6,
   FWI +0,4 »), ce qui est exactement ce qu'attend un opérationnel
4. Docker, CI/CD, DVC

**Une piste d'allègement, à ne tenter qu'en la mesurant** : retirer
`jours_depuis_dernier_feu` (jumelle à −0,999 de `feux_commune_365j`), `mois`,
`sin_mois`, `cos_mois` (doublons de `doy`) ferait passer de 52 à 48 features
sans perte attendue. À vérifier sur le découpage interne — en se rappelant
qu'il sous-estime les écarts d'un facteur 6 à 8.